# 06 — Modelagem
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** parte das 41 variáveis e da separação treino/teste do 05, treina e compara seis modelos de
churn, escolhe o melhor e o seu ponto de corte, mede o desempenho nos clientes guardados para a prova, empacota limpeza
+ modelo num pipeline de produção, publica as previsões no Catálogo e registra o modelo no MLflow.

**Origem:** é o notebook `modelo-antonio`, com **o mesmo código**, reorganizado no padrão dos outros notebooks:
perguntas de negócio, teoria antes de cada etapa, "como ler" depois de cada resultado e checklist. A preparação e os
testes estatísticos dele estão no 05; aqui a seção 1 só reconstrói as mesmas bases a partir das tabelas que o 05
gravou. A documentação completa do modelo está em `DOC_MODELO.md`.

**O modelo do Antonio, explícito:**

| O que é do Antonio | Onde |
|---|---|
| A preparação, os testes estatísticos e as 41 variáveis | 05, seções 2 e 3 |
| Os seis modelos, as grades de busca e a validação cruzada | 2.1 a 2.5 |
| O limiar pelo máximo do F1 fora da dobra e a regra de consistência que escolhe o modelo | 2.7 e 2.9 |
| A avaliação no teste, o KS, o lift e a leitura de negócio | Seções 3 a 5 |
| O pipeline de produção, as tabelas `_scored` e o registro no MLflow | Seção 6 |

**O que o grupo ajustou, sem mudar o método:** (1) o treino é ordenado por `customerID` antes da separação, que fica
gravada no 05 (os números se repetem em qualquer conta); (2) a busca roda sem processos paralelos (`n_jobs=1`), porque
no Serverless eles travam; (3) o MLflow grava os parâmetros do modelo escolhido, em vez de valores fixos; (4) as
respostas às perguntas de negócio e os "como ler".

**Qual modelo saiu:** na execução original do Antonio, a regra de consistência escolheu a **Ridge** (C = 0,03, limiar
0,4944, AUC 0,845 no teste). Nesta execução, com a separação ordenada, a mesma regra escolheu a **Lasso** (C = 0,3,
limiar 0,5046, AUC 0,830 no teste), com a Ridge a 0,003 de distância. O método e a regra são os dele; muda só qual das
duas logísticas, praticamente empatadas, fica em primeiro.

| # | Pergunta de negócio | Etapa |
|---|---|---|
| M1 | Que variáveis entram no modelo, e por quê? | Notebook 05 (aqui, só a reconstrução: seção 1) |
| M2 | Qual algoritmo prevê melhor o churn? | 2. Comparação de seis modelos |
| M3 | A partir de que probabilidade dizemos que o cliente vai sair? | 2. Limiar de decisão |
| M4 | O modelo funciona em clientes que ele não viu? | 3. Base de teste e consistência |
| M5 | Se só der para ligar para parte da carteira, quanto ganhamos? | 4. KS e lift |
| M6 | O que mais aumenta e diminui o risco de sair? | 5. Leitura de negócio |
| M7 | Quantos clientes da carteira estão em risco, e onde está o resultado? | 6. Produção, tabelas e MLflow |

| Notebook | Papel |
|---|---|
| 05 — Seleção de variáveis | Separação treino/teste e as 41 variáveis; grava `gold_churn_split` e `gold_churn_variaveis_modelo` |
| **06 — Modelagem** | **Seis modelos, limiar, avaliação, pipeline; grava `gold_churn_abt_scored` e `gold_churn_inferencia_scored`** |
| 07 — Segmentação | Lê `gold_churn_inferencia_scored` e cruza risco, valor e segmento |
| 08 — Entrega | Gera o `prediction` da entrega a partir de `gold_churn_inferencia_scored` |

## Glossário deste notebook

| Termo | Por extenso | O que é, em uma frase |
|---|---|---|
| **Holdout (base de teste)** | — | 20% dos clientes separados no início e usados só na avaliação final: a prova de que o modelo funciona em quem ele não viu |
| **Validação cruzada (CV)** | *Cross-validation* | Divide o treino em 5 partes, treina com 4 e mede na quinta, 5 vezes |
| **OOF** | *Out-of-fold* (fora da dobra) | A probabilidade de cada cliente dada por um modelo que não o viu no treino |
| **GridSearch / RandomizedSearch** | Busca em grade / busca aleatória | Testam combinações de hiperparâmetros na validação cruzada e ficam com a melhor |
| **Lasso, Ridge, Elastic Net** | Regularização L1, L2 e as duas juntas | Encolhem os coeficientes da logística para ela não decorar o treino; o Lasso zera os fracos |
| **Random forest, XGBoost, CatBoost** | — | Modelos de árvores combinadas; pegam interações e curvas sozinhos |
| **`class_weight="balanced"`, `scale_pos_weight`** | Peso das classes | Dão mais peso aos que saem (26,7% da base); aumentam o recall |
| **Precisão, recall, F1** | — | Dos marcados como churn, quantos saem; dos que saem, quantos são marcados; a média harmônica das duas |
| **ROC-AUC** | Área sob a curva ROC | Chance de o modelo dar nota maior a quem saiu do que a quem ficou (0,5 = sorteio) |
| **PR-AUC** | Área sob a curva precisão × recall | A mesma ideia olhando só a classe que importa; a linha de base é a taxa de churn |
| **KS** | Estatística de Kolmogorov-Smirnov | A maior distância entre as distribuições de nota de quem sai e de quem fica |
| **Lift** | — | Quantas vezes a taxa de churn de um grupo é maior que a média |
| **MCC, kappa** | Correlação de Matthews, kappa de Cohen | Acerto do Sim/Não descontado o acaso; bons para classes desbalanceadas |
| **Brier, log loss** | — | Erro da probabilidade (quadrático e logarítmico); menor é melhor |
| **Pearson, qui-quadrado, Fisher, Mann-Whitney** | Testes estatísticos | Associação de cada variável com o churn (contínua, binária, binária com poucos casos, contínua sem supor normalidade) |
| **V de Cramér** | — | Tamanho do efeito de uma associação entre categorias (0 = nenhuma, 1 = total) |
| **FDR (Benjamini-Hochberg)** | *False Discovery Rate* | Correção do p-valor quando se fazem muitos testes, para não "achar" efeito por acaso |
| **Pipeline, MLflow** | — | Limpeza + modelo num objeto só; registro do modelo para usar em qualquer lugar |
| **D1 a D25** | Decisões do checklist | Cada decisão do grupo, com o status |

## Checklist de decisões

**A mesma tabela em todos os notebooks.** ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## 0. Bibliotecas

O modelo usa XGBoost, CatBoost, statsmodels (correção para múltiplos testes) e seaborn, que não vêm instalados no
Serverless. Instalo tudo numa célula só e reinicio o Python (as bibliotecas só ficam disponíveis depois do reinício).
Depois, o `%run ./_comum` traz o `FQ`, o `virg` e o limite de threads que evita travar o Serverless.

In [ ]:
%pip install xgboost catboost statsmodels seaborn

In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./_comum

## 1. Carregar a gold e reconstruir as bases do 05

**O que faço:** leio a ABT, a separação treino/teste e as listas de variáveis que o 05 gravou, e refaço os mesmos
passos (retirar as redundantes, criar as dummies, retirar as 12 variáveis). Não repito os testes estatísticos: eles
estão no 05.

**Por que não sortear de novo:** a separação depende da ordem das linhas que o Spark entrega. Lendo a separação gravada,
o treino e o teste são **sempre os mesmos clientes, na mesma ordem**, e os números deste notebook não mudam de uma
execução para outra.

In [ ]:
# 1.1 Carrega a gold e reconstrói as bases do 05: mesma separação, mesma ordem, mesmas 41 colunas
assert "FQ" in globals(), "rode antes a célula %run ./_comum"
treino = spark.table(f"{FQ}.gold_churn_treino").toPandas().sort_values("customerID").reset_index(drop=True)
inferencia = spark.table(f"{FQ}.gold_churn_inferencia").toPandas().sort_values("customerID").reset_index(drop=True)
separacao = spark.table(f"{FQ}.gold_churn_split").toPandas()
listas = spark.table(f"{FQ}.gold_churn_variaveis_modelo").toPandas().sort_values(["tipo", "ordem"])
lista = lambda tipo: listas.loc[listas["tipo"] == tipo, "variavel"].tolist()
variaveis_redundantes, variaveis_categoricas = lista("redundante"), lista("categorica_dummy")
variaveis_retirar, colunas_finais = lista("retirada"), lista("modelo")

# os mesmos passos do 05 (2.1, 2.3 e 3.1), na mesma ordem
df_limpo = treino.drop(columns=variaveis_redundantes).copy()
df_modelagem = pd.get_dummies(df_limpo, columns=variaveis_categoricas, drop_first=True, dtype=int)
X = df_modelagem.drop(columns=["Churn"]).copy()
y = pd.to_numeric(df_modelagem["Churn"], errors="raise").astype(int)

posicao = pd.Series(treino.index, index=treino["customerID"])
def linhas_da(base):
    s = separacao[separacao["base"] == base].sort_values("ordem")
    return posicao.loc[s["customerID"]].to_numpy()
i_treino, i_teste = linhas_da("Treino"), linhas_da("Teste")
X_treino_original, X_teste_original = X.loc[i_treino], X.loc[i_teste]
y_treino, y_teste = y.loc[i_treino], y.loc[i_teste]

X_treino = X_treino_original.select_dtypes(include=["number", "bool"]).drop(columns=["customerID", "Churn"], errors="ignore").astype(float)
X_teste = X_teste_original.select_dtypes(include=["number", "bool"]).drop(columns=["customerID", "Churn"], errors="ignore").astype(float)
X_teste = X_teste.reindex(columns=X_treino.columns)
X_treino_limpo = X_treino.drop(columns=variaveis_retirar, errors="ignore").copy()
X_teste_limpo = X_teste.drop(columns=variaveis_retirar, errors="ignore").copy()

conferencias = [
    ("todo cliente do treino da gold está em uma, e só uma, das partes", len(i_treino) + len(i_teste) == len(treino) and not set(i_treino) & set(i_teste)),
    ("4.507 no treino e 1.127 no teste", (len(i_treino), len(i_teste)) == (4507, 1127)),
    ("as 41 colunas, na ordem gravada pelo 05", X_treino_limpo.columns.tolist() == colunas_finais),
    ("mesmas colunas no treino e no teste", X_treino_limpo.columns.equals(X_teste_limpo.columns)),
    ("taxa de churn igual nas duas partes (estratificação)", abs(y_treino.mean() - y_teste.mean()) < 0.002),
]
display(pd.DataFrame(conferencias, columns=["conferência", "ok"]))
assert all(ok for _, ok in conferencias), "as bases não batem com o 05: rodar de novo o notebook 05"
print(f"Treino: {X_treino_limpo.shape} · teste: {X_teste_limpo.shape} · churn no treino {virg(100 * y_treino.mean(), 2)}% "
      f"e no teste {virg(100 * y_teste.mean(), 2)}%")

> **Como ler:** as cinco conferências precisam dar `True`. Se alguma falhar, o 05 não rodou até a 3.13 nesta
> conta (ou rodou com outra gold): rodar o 05 de novo antes de seguir.

## 2. Comparação de seis modelos e limiar de decisão

> **M2 — Qual algoritmo prevê melhor o churn?**
> **M3 — A partir de que probabilidade dizemos que o cliente vai sair?**

### Teoria: modelos, busca de hiperparâmetros e critério

| Família | Modelos | Busca |
|---|---|---|
| Regressão logística regularizada | Lasso (L1), Ridge (L2), Elastic Net | Busca em grade de `C` (e de `l1_ratio` no Elastic Net) |
| Árvores combinadas | *Random forest*, XGBoost, CatBoost | Busca aleatória, 40 combinações |

- **Validação cruzada de 5 dobras** no treino; o critério da busca é o **F1** (equilíbrio entre precisão e recall),
  com precisão, recall, ROC-AUC e PR-AUC registrados.
- **Desbalanceamento:** com 26,7% de churn, as logísticas usam classes balanceadas e as árvores, `scale_pos_weight`.
- **Limiar de decisão (M3):** não é fixo em 0,5. Para cada modelo, calculo as probabilidades **fora da dobra** no treino e
  escolho o limiar que **maximiza o F1** nelas. O limiar fica congelado e só depois é aplicado ao teste.
- **Escolha do modelo:** pela **consistência** entre treino (fora da dobra), teste e base total:
  Score = F1 médio − amplitude do F1. Penaliza o modelo que vai bem numa base e mal em outra.

In [ ]:
# 2.1 Bibliotecas dos modelos
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_predict
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from scipy.stats import randint, uniform, loguniform

In [ ]:
# 2.2 Confere as bases e garante números e Churn 0/1
# ============================================================
# 2. CONFERÊNCIA DAS BASES
# ============================================================

# Utilizaremos as bases já preparadas e com as variáveis
# redundantes removidas.

assert X_treino_limpo.columns.equals(X_teste_limpo.columns), (
    "As bases de treino e teste precisam ter as mesmas colunas."
)

assert len(X_treino_limpo) == len(y_treino)
assert len(X_teste_limpo) == len(y_teste)

# Garantir que todas as variáveis preditoras sejam numéricas.

X_treino_limpo = X_treino_limpo.apply(
    pd.to_numeric,
    errors="coerce"
).replace([np.inf, -np.inf], np.nan)

X_teste_limpo = X_teste_limpo.apply(
    pd.to_numeric,
    errors="coerce"
).replace([np.inf, -np.inf], np.nan)

# Garantir que a variável-alvo esteja codificada como 0 e 1.

y_treino = pd.to_numeric(y_treino, errors="raise").astype(int)
y_teste = pd.to_numeric(y_teste, errors="raise").astype(int)

assert set(y_treino.unique()).issubset({0, 1})
assert set(y_teste.unique()).issubset({0, 1})

print("=" * 65)
print("CONFERÊNCIA DAS BASES")
print("=" * 65)

print("Treino:", X_treino_limpo.shape)
print("Teste:", X_teste_limpo.shape)

print("\nDistribuição de Churn no treino:")
print(y_treino.value_counts(normalize=True).round(4))

print("\nDistribuição de Churn no teste:")
print(y_teste.value_counts(normalize=True).round(4))

In [ ]:
# 2.3 Validação cruzada de 5 dobras e métricas
# ============================================================
# 3. CONFIGURAÇÃO DA VALIDAÇÃO CRUZADA
# ============================================================

# Utilizaremos 5 folds estratificados.
#
# Em cada rodada:
# - 4 partes serão utilizadas para treinamento.
# - 1 parte será utilizada para validação.
#
# O processo será repetido 5 vezes.

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Avaliar os modelos utilizando três métricas.
#
# O F1-score será utilizado como critério principal
# para escolher os melhores hiperparâmetros.

# Além do F1 (critério principal), avaliaremos ROC-AUC e PR-AUC,
# que são independentes do limiar de decisão.

metricas = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [ ]:
# 2.4 Os seis modelos e as grades de hiperparâmetros
# ============================================================
# 4. DEFINIÇÃO DOS MODELOS E HIPERPARÂMETROS
# ============================================================

modelos = {}


# ------------------------------------------------------------
# MODELO 1 — REGRESSÃO LOGÍSTICA LASSO (L1)
# ------------------------------------------------------------

pipeline_lasso = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=10000,
        class_weight="balanced",
        random_state=42
    ))
])

# C = 0.001 removido: regularização excessiva zera as previsões
# da classe positiva em alguns folds (warning de precision indefinida).
# Grade ampliada com valores intermediários para uma busca mais fina.

parametros_lasso = {
    "modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]
}

modelos["Logistica_Lasso"] = {
    "estimador": pipeline_lasso,
    "parametros": parametros_lasso,
    "tipo": "grid"
}


# ------------------------------------------------------------
# MODELO 2 — REGRESSÃO LOGÍSTICA RIDGE (L2)
# ------------------------------------------------------------

pipeline_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=10000,
        class_weight="balanced",
        random_state=42
    ))
])

parametros_ridge = {
    "modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]
}

modelos["Logistica_Ridge"] = {
    "estimador": pipeline_ridge,
    "parametros": parametros_ridge,
    "tipo": "grid"
}


# ------------------------------------------------------------
# MODELO 3 — REGRESSÃO LOGÍSTICA ELASTIC NET
# ------------------------------------------------------------

pipeline_elasticnet = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=10000,
        class_weight="balanced",
        random_state=42
    ))
])

parametros_elasticnet = {
    "modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100],
    "modelo__l1_ratio": [0.10, 0.25, 0.50, 0.75, 0.90]
}

modelos["Logistica_ElasticNet"] = {
    "estimador": pipeline_elasticnet,
    "parametros": parametros_elasticnet,
    "tipo": "grid"
}


# ------------------------------------------------------------
# MODELO 4 — RANDOM FOREST
# ------------------------------------------------------------

pipeline_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("modelo", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

parametros_rf = {
    "modelo__n_estimators": randint(200, 601),
    "modelo__max_depth": [None, 3, 5, 8, 10, 15, 20, 30],
    "modelo__min_samples_split": randint(2, 31),
    "modelo__min_samples_leaf": randint(1, 16),
    "modelo__max_features": ["sqrt", "log2", 0.5, 0.7],
    "modelo__class_weight": ["balanced", "balanced_subsample"]
}

modelos["Random_Forest"] = {
    "estimador": pipeline_rf,
    "parametros": parametros_rf,
    "tipo": "random",
    "n_iter": 40
}


# ------------------------------------------------------------
# MODELO 5 — XGBOOST
# ------------------------------------------------------------

# Calcular o peso da classe positiva utilizando
# somente a variável-alvo do conjunto de treino.

# float(...) é essencial: o valor bruto é um np.float64, que
# faz o sklearn falhar ao clonar o CatBoostClassifier durante
# a busca de hiperparâmetros (e afeta também os grids abaixo).

peso_positivo = float(
    (y_treino == 0).sum() /
    (y_treino == 1).sum()
)

pipeline_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("modelo", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
        scale_pos_weight=peso_positivo,
        verbosity=0
    ))
])

parametros_xgb = {
    "modelo__n_estimators": randint(100, 601),
    "modelo__max_depth": randint(2, 10),
    "modelo__learning_rate": loguniform(0.005, 0.3),
    "modelo__subsample": uniform(0.5, 0.5),
    "modelo__colsample_bytree": uniform(0.5, 0.5),
    "modelo__min_child_weight": randint(1, 11),
    "modelo__gamma": uniform(0, 5),
    "modelo__reg_alpha": loguniform(0.001, 10),
    "modelo__reg_lambda": loguniform(0.1, 10),
    "modelo__scale_pos_weight": [peso_positivo, 1.0]
}

modelos["XGBoost"] = {
    "estimador": pipeline_xgb,
    "parametros": parametros_xgb,
    "tipo": "random",
    "n_iter": 40
}


# ------------------------------------------------------------
# MODELO 6 — CATBOOST
# ------------------------------------------------------------

pipeline_catboost = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("modelo", CatBoostClassifier(
        random_seed=42,
        logging_level="Silent",
        allow_writing_files=False,
        scale_pos_weight=peso_positivo
    ))
])

# IMPORTANTE: aqui usamos listas com valores nativos do Python
# (e não as distribuições randint/loguniform/uniform, que amostram
# np.int64/np.float64). O CatBoost não reproduz escalares numpy no
# clone() do sklearn, o que faz o GridSearch/CrossVal falhar.
# O RandomizedSearch continua sorteando 40 combinações, no mesmo
# padrão dos outros modelos.

parametros_catboost = {
    "modelo__iterations": [100, 150, 200, 250, 300, 350, 400, 450, 500],
    "modelo__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30],
    "modelo__depth": [4, 5, 6, 7, 8],
    "modelo__l2_leaf_reg": [1.0, 3.0, 5.0, 7.0, 9.0],
    "modelo__subsample": [0.7, 0.8, 0.9, 1.0],
    "modelo__border_count": [32, 64, 128],
    "modelo__scale_pos_weight": [peso_positivo, 1.0]
}

modelos["CatBoost"] = {
    "estimador": pipeline_catboost,
    "parametros": parametros_catboost,
    "tipo": "random",
    "n_iter": 40
}

In [ ]:
# 2.5 Treina e busca os hiperparâmetros (é a célula mais demorada)
# ajuste do grupo: n_jobs=1 (um processo só), porque no Serverless os processos paralelos travam;
# o resultado é o mesmo, só muda o tempo. O print do tempo mostra que a célula não travou.
import time
inicio_busca = time.time()
# ============================================================
# 5. TREINAMENTO E VALIDAÇÃO CRUZADA DOS CINCO MODELOS
# ============================================================

resultados = {}
resultados_cv = []

print("\n" + "=" * 65)
print("INICIANDO VALIDAÇÃO CRUZADA")
print("=" * 65)

for nome, configuracao in modelos.items():

    print(f"\nTreinando modelo: {nome} ({time.time() - inicio_busca:.0f} s até aqui)", flush=True)

    # Grid Search para regressões logísticas.

    if configuracao["tipo"] == "grid":

        busca = GridSearchCV(
            estimator=configuracao["estimador"],
            param_grid=configuracao["parametros"],
            scoring=metricas,
            refit="f1",
            cv=cv,
            n_jobs=1,
            verbose=0,
            return_train_score=False
        )

    # Randomized Search para Random Forest e XGBoost.

    else:

        busca = RandomizedSearchCV(
            estimator=configuracao["estimador"],
            param_distributions=configuracao["parametros"],
            n_iter=configuracao["n_iter"],
            scoring=metricas,
            refit="f1",
            cv=cv,
            random_state=42,
            n_jobs=1,
            verbose=0,
            return_train_score=False
        )

    # Treinar e otimizar utilizando somente a base de treino.
    # A validação cruzada separa internamente os folds.

    busca.fit(X_treino_limpo, y_treino)

    # Armazenar o melhor modelo encontrado.

    resultados[nome] = busca

    # Recuperar os resultados da melhor combinação.

    indice = busca.best_index_

    resultados_cv.append({
        "Modelo": nome,
        "Precision_CV": busca.cv_results_["mean_test_precision"][indice],
        "Recall_CV": busca.cv_results_["mean_test_recall"][indice],
        "F1_CV": busca.cv_results_["mean_test_f1"][indice],
        "F1_CV_std": busca.cv_results_["std_test_f1"][indice],
        "ROC_AUC_CV": busca.cv_results_["mean_test_roc_auc"][indice],
        "PR_AUC_CV": busca.cv_results_["mean_test_pr_auc"][indice],
        "Melhores_parametros": busca.best_params_
    })

    print("Melhor F1 médio:", round(busca.best_score_, 4))
    print("Melhores parâmetros:", busca.best_params_)
    print(f"Tempo até aqui: {(time.time() - inicio_busca) / 60:.1f} min", flush=True)

In [ ]:
# 2.6 Resultados da validação cruzada
# ============================================================
# 6. COMPARAÇÃO DOS RESULTADOS DA VALIDAÇÃO CRUZADA
# ============================================================

df_resultados_cv = pd.DataFrame(resultados_cv)

df_resultados_cv = df_resultados_cv.sort_values(
    "F1_CV",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("RESULTADOS DA VALIDAÇÃO CRUZADA")
print("=" * 65)

display(
    df_resultados_cv[
        [
            "Modelo",
            "Precision_CV",
            "Recall_CV",
            "F1_CV",
            "F1_CV_std",
            "ROC_AUC_CV",
            "PR_AUC_CV"
        ]
    ].round(4)
)

> **Como ler a validação cruzada:** os seis modelos ficam muito próximos (ROC-AUC entre 0,841 e 0,847; F1 entre 0,633 e
> 0,640). A diferença entre o primeiro e o último no F1 (0,007) é menor que a variação entre as dobras (desvio de 0,016
> a 0,024): pela validação cruzada, estão empatados. As três logísticas lideram por pouco; as árvores não ganham, porque
> as relações aqui são quase lineares e a interação mais forte (cliente novo no contrato mensal) já está numa coluna.

In [ ]:
# 2.7 Limiar do maior F1 (fora da dobra) e avaliação no treino, no teste e na base total
# ============================================================
# 7. AVALIAÇÃO — TREINO (OOF), TESTE E BASE TOTAL
# ============================================================

# Utilizar os melhores modelos encontrados na validação cruzada
# para realizar previsões na base de teste e na base total.
#
# O limiar de decisão não é mais fixo em 0.5: para cada modelo,
# o limiar ótimo é definido nas previsões out-of-fold do treino
# e só depois aplicado às bases de teste e total.
#
# IMPORTANTE: nenhum modelo é retreinado aqui. O modelo definido
# na base de treino é apenas aplicado à base de teste e à base
# completa (treino + teste), para verificar a consistência dos
# seus resultados.

resultados_teste = []

# Base total (treino + teste juntos): o modelo treinado apenas
# na base de treino é aplicado a ela, sem qualquer retreino.

X_total = pd.concat([X_treino_limpo, X_teste_limpo], axis=0)
y_total = pd.concat([y_treino, y_teste], axis=0)

print("\n" + "=" * 65)
print("AVALIAÇÃO — TREINO (OOF), TESTE E TOTAL")
print("=" * 65)

for nome, busca in resultados.items():

    melhor_modelo = busca.best_estimator_

    # 1. Probabilidades out-of-fold na base de treino
    #    (o limiar é definido sem tocar na base de teste).

    probas_oof = cross_val_predict(
        melhor_modelo,
        X_treino_limpo,
        y_treino,
        cv=cv,
        method="predict_proba",
        n_jobs=1   # ajuste do grupo: sem processos paralelos no Serverless
    )[:, 1]

    # 2. Limiar que maximiza o F1 nas previsões out-of-fold.

    prec, rec, limiares = precision_recall_curve(
        y_treino,
        probas_oof
    )

    f1_por_limiar = (2 * prec * rec) / (prec + rec + 1e-12)

    if len(limiares) == 0:
        limiar_otimo = 0.5
    else:
        limiar_otimo = limiares[np.argmax(f1_por_limiar[:-1])]

    # 3. Probabilidades na base de teste e classes no limiar otimizado.

    y_proba = melhor_modelo.predict_proba(X_teste_limpo)[:, 1]

    y_pred = (y_proba >= limiar_otimo).astype(int)

    # Calcular métricas.

    precision = precision_score(
        y_teste,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_teste,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_teste,
        y_pred,
        zero_division=0
    )

    # Desempenho no próprio treino (out-of-fold), para comparar
    # com o teste e detectar sobreajuste.

    y_pred_oof = (probas_oof >= limiar_otimo).astype(int)

    precision_oof = precision_score(y_treino, y_pred_oof, zero_division=0)
    recall_oof = recall_score(y_treino, y_pred_oof, zero_division=0)
    f1_treino_oof = f1_score(y_treino, y_pred_oof, zero_division=0)

    # Aplicar o modelo treinado na base de treino à base total
    # (treino + teste), no mesmo limiar, apenas para verificar a
    # consistência — sem retreinar e sem recalcular parâmetros.

    y_proba_total = melhor_modelo.predict_proba(X_total)[:, 1]
    y_pred_total = (y_proba_total >= limiar_otimo).astype(int)

    precision_total = precision_score(y_total, y_pred_total, zero_division=0)
    recall_total = recall_score(y_total, y_pred_total, zero_division=0)
    f1_total = f1_score(y_total, y_pred_total, zero_division=0)

    # Métricas independentes de limiar.

    roc_auc = roc_auc_score(y_teste, y_proba)
    pr_auc = average_precision_score(y_teste, y_proba)

    resultados_teste.append({
        "Modelo": nome,
        "Limiar_Otimo": limiar_otimo,
        "Precision_Treino_OOF": precision_oof,
        "Recall_Treino_OOF": recall_oof,
        "F1_Treino_OOF": f1_treino_oof,
        "Precision_Teste": precision,
        "Recall_Teste": recall,
        "F1_Teste": f1,
        "Precision_Total": precision_total,
        "Recall_Total": recall_total,
        "F1_Total": f1_total,
        "Gap_Treino_Teste": f1_treino_oof - f1,
        "ROC_AUC_Teste": roc_auc,
        "PR_AUC_Teste": pr_auc
    })

    print(f"\nModelo: {nome}")
    print(f"Limiar otimizado (definido no treino): {limiar_otimo:.4f}")

    print("\nMatriz de confusão:")
    print(confusion_matrix(y_teste, y_pred))

    print("\nRelatório de classificação:")
    print(
        classification_report(
            y_teste,
            y_pred,
            target_names=["Não Churn", "Churn"],
            zero_division=0
        )
    )

In [ ]:
# 2.8 Tabela final de comparação
# ============================================================
# 8. TABELA FINAL DE COMPARAÇÃO
# ============================================================

df_resultados_teste = pd.DataFrame(resultados_teste)

# Unir resultados de validação cruzada e teste.

df_comparacao = df_resultados_cv.merge(
    df_resultados_teste,
    on="Modelo"
)

# Ordenar pelo F1 da validação cruzada como referência.
# A escolha definitiva do melhor modelo é feita na seção 9,
# pelas métricas de F1, recall e precision e pela consistência
# entre as bases de treino, teste e total.

df_comparacao = df_comparacao.sort_values(
    "F1_CV",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("COMPARAÇÃO FINAL DOS CINCO MODELOS")
print("=" * 65)

display(
    df_comparacao[
        [
            "Modelo",
            "Precision_CV",
            "Recall_CV",
            "F1_CV",
            "F1_CV_std",
            "ROC_AUC_CV",
            "PR_AUC_CV",
            "Limiar_Otimo",
            "Precision_Treino_OOF",
            "Recall_Treino_OOF",
            "F1_Treino_OOF",
            "Precision_Teste",
            "Recall_Teste",
            "F1_Teste",
            "Precision_Total",
            "Recall_Total",
            "F1_Total",
            "Gap_Treino_Teste",
            "ROC_AUC_Teste",
            "PR_AUC_Teste"
        ]
    ].round(4)
)

In [ ]:
# 2.9 Escolha do melhor modelo pela consistência (F1 médio − amplitude)
# ============================================================
# 9. IDENTIFICAR O MELHOR MODELO — F1, RECALL, PRECISION
#    E CONSISTÊNCIA ENTRE TREINO, TESTE E TOTAL
# ============================================================

# Critério: o melhor modelo é aquele com as melhores métricas
# de F1, recall e precision E resultados consistentes nas três
# bases (treino OOF, teste e total).
#
# - O F1 médio entre as bases mede a qualidade global.
# - A amplitude (máximo − mínimo) entre as bases mede a
#   consistência: modelos sobreajustados têm F1 alto no
#   treino/total e baixo no teste, gerando amplitude grande.
#
# Score final = F1 médio − amplitude do F1.
# Modelos bons e consistentes maximizam esse score.

colunas_precision = [
    "Precision_Treino_OOF", "Precision_Teste", "Precision_Total"
]
colunas_recall = [
    "Recall_Treino_OOF", "Recall_Teste", "Recall_Total"
]
colunas_f1 = ["F1_Treino_OOF", "F1_Teste", "F1_Total"]

df_consistencia = df_comparacao[[
    "Modelo",
    *colunas_precision,
    *colunas_recall,
    *colunas_f1
]].copy()

df_consistencia["Precision_Medio"] = df_consistencia[colunas_precision].mean(axis=1)
df_consistencia["Recall_Medio"] = df_consistencia[colunas_recall].mean(axis=1)
df_consistencia["F1_Medio"] = df_consistencia[colunas_f1].mean(axis=1)

df_consistencia["Amplitude_Precision"] = (
    df_consistencia[colunas_precision].max(axis=1)
    - df_consistencia[colunas_precision].min(axis=1)
)
df_consistencia["Amplitude_Recall"] = (
    df_consistencia[colunas_recall].max(axis=1)
    - df_consistencia[colunas_recall].min(axis=1)
)
df_consistencia["Amplitude_F1"] = (
    df_consistencia[colunas_f1].max(axis=1)
    - df_consistencia[colunas_f1].min(axis=1)
)

df_consistencia["Score"] = (
    df_consistencia["F1_Medio"] - df_consistencia["Amplitude_F1"]
)

df_consistencia = df_consistencia.sort_values(
    "Score",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("CONSISTÊNCIA ENTRE TREINO (OOF), TESTE E TOTAL")
print("=" * 65)

display(df_consistencia.round(4))

melhor_nome = df_consistencia.iloc[0]["Modelo"]

linha_melhor = df_comparacao.loc[
    df_comparacao["Modelo"] == melhor_nome
].iloc[0]

linha_melhor_consistencia = df_consistencia.iloc[0]

limiar_melhor = linha_melhor["Limiar_Otimo"]

print("=" * 65)
print("MELHOR MODELO — MÉTRICAS E CONSISTÊNCIA")
print("=" * 65)

print("Modelo escolhido:", melhor_nome)
print(f"F1 médio nas três bases: {linha_melhor_consistencia['F1_Medio']:.4f}")
print(f"Amplitude do F1 entre as bases: {linha_melhor_consistencia['Amplitude_F1']:.4f}")
print(f"Amplitude do precision entre as bases: {linha_melhor_consistencia['Amplitude_Precision']:.4f}")
print(f"Amplitude do recall entre as bases: {linha_melhor_consistencia['Amplitude_Recall']:.4f}")

print(
    f"\nPrecision — treino (OOF): {linha_melhor['Precision_Treino_OOF']:.4f}"
    f" | teste: {linha_melhor['Precision_Teste']:.4f}"
    f" | total: {linha_melhor['Precision_Total']:.4f}"
)
print(
    f"Recall    — treino (OOF): {linha_melhor['Recall_Treino_OOF']:.4f}"
    f" | teste: {linha_melhor['Recall_Teste']:.4f}"
    f" | total: {linha_melhor['Recall_Total']:.4f}"
)
print(
    f"F1        — treino (OOF): {linha_melhor['F1_Treino_OOF']:.4f}"
    f" | teste: {linha_melhor['F1_Teste']:.4f}"
    f" | total: {linha_melhor['F1_Total']:.4f}"
)

print(f"\nLimiar otimizado (definido no treino): {limiar_melhor:.4f}")

print(
    "\nHiperparâmetros (definidos na base de treino, congelados):",
    resultados[melhor_nome].best_params_
)

> **Como ler a escolha e respostas (M2 e M3)**
> - **Consistência:** a Lasso tem o maior Score (0,617): F1 de 0,641 no treino (fora da dobra), 0,624 no teste e 0,637
>   na base total, amplitude de 0,017. A Ridge fica logo atrás (0,615), um empate técnico. As árvores inflam o F1 na
>   base total (0,69 a 0,71, sinal de sobreajuste) e ficam por último.
> - **M2:** a **regressão logística Lasso** (C = 0,3, classes balanceadas, 41 variáveis) é o melhor modelo.
> - **M3:** o limiar é **0,5046**, o que maximiza o F1 nas probabilidades fora da dobra do treino.
> - **Mudança em relação à execução original:** com a separação antiga, a Ridge (C = 0,03) tinha vencido. Com o treino
>   ordenado por `customerID` (para os números se repetirem em qualquer conta), os clientes do teste mudaram e a Lasso
>   passou à frente por 0,003. A regra é a mesma; a decisão segue a regra (D21).

In [ ]:
# 2.10 Variáveis que o Lasso manteve (coeficiente diferente de zero)
# ============================================================
# 10. IDENTIFICAR VARIÁVEIS SELECIONADAS PELO LASSO
# ============================================================

# O Lasso pode zerar coeficientes.
# Vamos identificar quais variáveis permaneceram
# com coeficientes diferentes de zero.

modelo_lasso = resultados["Logistica_Lasso"].best_estimator_

coef_lasso = modelo_lasso.named_steps["modelo"].coef_[0]

df_coef_lasso = pd.DataFrame({
    "Variavel": X_treino_limpo.columns,
    "Coeficiente": coef_lasso
})

df_coef_lasso["Importancia_absoluta"] = (
    df_coef_lasso["Coeficiente"].abs()
)

df_coef_lasso["Selecionada"] = (
    df_coef_lasso["Coeficiente"] != 0
)

df_coef_lasso = df_coef_lasso.sort_values(
    "Importancia_absoluta",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("SELEÇÃO DE VARIÁVEIS — LASSO")
print("=" * 65)

print(
    "Quantidade de variáveis selecionadas:",
    df_coef_lasso["Selecionada"].sum()
)

print(
    "Quantidade de variáveis com coeficiente zero:",
    (~df_coef_lasso["Selecionada"]).sum()
)

display(df_coef_lasso)

In [ ]:
# 2.11 Variáveis que o Elastic Net manteve
# ============================================================
# 11. IDENTIFICAR VARIÁVEIS SELECIONADAS PELO ELASTIC NET
# ============================================================

modelo_elastic = resultados[
    "Logistica_ElasticNet"
].best_estimator_

coef_elastic = modelo_elastic.named_steps["modelo"].coef_[0]

df_coef_elastic = pd.DataFrame({
    "Variavel": X_treino_limpo.columns,
    "Coeficiente": coef_elastic
})

df_coef_elastic["Importancia_absoluta"] = (
    df_coef_elastic["Coeficiente"].abs()
)

df_coef_elastic["Selecionada"] = (
    df_coef_elastic["Coeficiente"] != 0
)

df_coef_elastic = df_coef_elastic.sort_values(
    "Importancia_absoluta",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("SELEÇÃO DE VARIÁVEIS — ELASTIC NET")
print("=" * 65)

print(
    "Quantidade de variáveis selecionadas:",
    df_coef_elastic["Selecionada"].sum()
)

print(
    "Quantidade de variáveis com coeficiente zero:",
    (~df_coef_elastic["Selecionada"]).sum()
)

display(df_coef_elastic)

> **Como ler o Lasso e o Elastic Net:** são uma segunda seleção de variáveis, feita pelo próprio modelo: os coeficientes
> zerados são as variáveis que pouco acrescentam às outras. A Lasso zera 11 das 41 (entre elas `TotalCharges`,
> `has_family`, `n_protecao` e `pagamento_automatico`, cobertas por outras colunas); o Elastic Net zera 6, todas entre as
> 11 da Lasso. Repare no preço: as duas ficam com `MonthlyCharges` positivo e `mensalidade_relativa_pacote` negativo, e
> quase zeram o tipo de internet. É a mesma multicolinearidade que a 5.5c do 05 mostrou: o preço carrega o tipo de
> internet, e os dois coeficientes se leem juntos.

## 3. Avaliação na base de teste

> **M4 — O modelo funciona em clientes que ele não viu?**

### Teoria: as métricas da avaliação final

Tudo nesta seção usa a **base de teste** (1.127 clientes, nunca usados nas escolhas) e o limiar congelado de 0,5046.
- **Ordenação (sem limiar):** ROC-AUC, PR-AUC e KS.
- **Sim/Não (com o limiar):** acurácia, acurácia balanceada, precisão, recall, especificidade, F1, MCC e kappa.
- **Probabilidade:** log loss e Brier.
- **Gráficos:** curva ROC, curva precisão × recall, calibração, matriz de confusão, ganho acumulado e distribuição das
  probabilidades por classe.

In [ ]:
# 3.1 Métricas completas do melhor modelo na base de teste
# ============================================================
# AVALIAÇÃO DETALHADA DO MELHOR MODELO
# Curvas ROC, PR, calibração, ganho cumulativo,
# matriz de confusão e métricas completas.
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_curve,
    log_loss,
    brier_score_loss,
    matthews_corrcoef,
    cohen_kappa_score
)
from sklearn.calibration import calibration_curve

# Melhor modelo treinado apenas na base de treino.
melhor_modelo = resultados[melhor_nome].best_estimator_

# Probabilidades e previsões na base de teste (holdout).
y_proba_melhor = melhor_modelo.predict_proba(X_teste_limpo)[:, 1]
y_pred_melhor = (y_proba_melhor >= limiar_melhor).astype(int)

# ------------------------------------------------------------
# MÉTRICAS COMPLETAS
# ------------------------------------------------------------

fpr, tpr, _ = roc_curve(y_teste, y_proba_melhor)
ks = np.max(tpr - fpr)

metricas_melhor = pd.DataFrame([
    {"Métrica": "Acurácia", "Valor": accuracy_score(y_teste, y_pred_melhor)},
    {"Métrica": "Acurácia balanceada", "Valor": balanced_accuracy_score(y_teste, y_pred_melhor)},
    {"Métrica": "Precision (classe Churn)", "Valor": precision_score(y_teste, y_pred_melhor, zero_division=0)},
    {"Métrica": "Recall (classe Churn)", "Valor": recall_score(y_teste, y_pred_melhor, zero_division=0)},
    {"Métrica": "Especificidade (classe Não Churn)", "Valor": recall_score(y_teste, y_pred_melhor, pos_label=0, zero_division=0)},
    {"Métrica": "F1-score (classe Churn)", "Valor": f1_score(y_teste, y_pred_melhor, zero_division=0)},
    {"Métrica": "ROC-AUC", "Valor": roc_auc_score(y_teste, y_proba_melhor)},
    {"Métrica": "PR-AUC (average precision)", "Valor": average_precision_score(y_teste, y_proba_melhor)},
    {"Métrica": "Estatística KS", "Valor": ks},
    {"Métrica": "Log loss", "Valor": log_loss(y_teste, y_proba_melhor)},
    {"Métrica": "Brier score", "Valor": brier_score_loss(y_teste, y_proba_melhor)},
    {"Métrica": "MCC (Matthews)", "Valor": matthews_corrcoef(y_teste, y_pred_melhor)},
    {"Métrica": "Cohen kappa", "Valor": cohen_kappa_score(y_teste, y_pred_melhor)}
])

print("=" * 65)
print(f"MÉTRICAS DO MELHOR MODELO — {melhor_nome} (base de teste)")
print("=" * 65)

display(metricas_melhor.round(4))

In [ ]:
# 3.2 Painel de gráficos da avaliação
import matplotlib.pyplot as plt
# ------------------------------------------------------------
# PAINEL DE GRÁFICOS
# ------------------------------------------------------------

fig, eixos = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"Avaliação do melhor modelo ({melhor_nome}) na base de teste", fontsize=14)

# 1. Curva ROC
eixos[0, 0].plot(fpr, tpr, color="darkorange", lw=2, label=f"Modelo (AUC = {roc_auc_score(y_teste, y_proba_melhor):.3f})")
eixos[0, 0].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Aleatório")
eixos[0, 0].set_xlabel("Taxa de falsos positivos")
eixos[0, 0].set_ylabel("Taxa de verdadeiros positivos")
eixos[0, 0].set_title("Curva ROC")
eixos[0, 0].legend(loc="lower right")
eixos[0, 0].grid(alpha=0.3)

# 2. Curva Precision-Recall
prec_vals, rec_vals, _ = precision_recall_curve(y_teste, y_proba_melhor)
eixos[0, 1].plot(rec_vals, prec_vals, color="darkgreen", lw=2, label=f"Modelo (AP = {average_precision_score(y_teste, y_proba_melhor):.3f})")
taxa_churn = y_teste.mean()
eixos[0, 1].axhline(taxa_churn, color="gray", lw=1, linestyle="--", label=f"Baseline ({taxa_churn:.3f})")
eixos[0, 1].set_xlabel("Recall")
eixos[0, 1].set_ylabel("Precision")
eixos[0, 1].set_title("Curva Precision-Recall")
eixos[0, 1].legend(loc="upper right")
eixos[0, 1].grid(alpha=0.3)

# 3. Calibração das probabilidades
fracao_positiva, prob_prevista = calibration_curve(y_teste, y_proba_melhor, n_bins=10, strategy="quantile")
eixos[0, 2].plot(prob_prevista, fracao_positiva, marker="o", lw=2, color="purple", label="Modelo")
eixos[0, 2].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Perfeitamente calibrado")
eixos[0, 2].set_xlabel("Probabilidade prevista")
eixos[0, 2].set_ylabel("Fração real de churn")
eixos[0, 2].set_title("Calibração das probabilidades")
eixos[0, 2].legend(loc="upper left")
eixos[0, 2].grid(alpha=0.3)

# 4. Matriz de confusão no limiar otimizado
cm = confusion_matrix(y_teste, y_pred_melhor)
im = eixos[1, 0].imshow(cm, cmap="Blues")
fig.colorbar(im, ax=eixos[1, 0])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        eixos[1, 0].text(j, i, f"{cm[i, j]}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
eixos[1, 0].set_xticks([0, 1])
eixos[1, 0].set_yticks([0, 1])
eixos[1, 0].set_xticklabels(["Não Churn", "Churn"])
eixos[1, 0].set_yticklabels(["Não Churn", "Churn"])
eixos[1, 0].set_xlabel("Previsto")
eixos[1, 0].set_ylabel("Real")
eixos[1, 0].set_title(f"Matriz de confusão (limiar = {limiar_melhor:.3f})")

# 5. Curva de ganho cumulativo
ordem = np.argsort(-y_proba_melhor)
churn_ordenado = y_teste.to_numpy()[ordem]
fracao_contatada = np.arange(1, len(churn_ordenado) + 1) / len(churn_ordenado)
ganho_cumulativo = np.cumsum(churn_ordenado) / churn_ordenado.sum()

eixos[1, 1].plot(fracao_contatada, ganho_cumulativo, color="crimson", lw=2, label="Modelo")
eixos[1, 1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Sem modelo")
eixos[1, 1].axvline(0.20, color="navy", lw=1, linestyle=":", alpha=0.7)
eixos[1, 1].set_xlabel("Fração da base contatada (ordenada pelo score)")
eixos[1, 1].set_ylabel("Fração de churners capturados")
eixos[1, 1].set_title("Curva de ganho cumulativo")
eixos[1, 1].legend(loc="lower right")
eixos[1, 1].grid(alpha=0.3)

captura_20 = ganho_cumulativo[int(np.ceil(0.20 * len(ganho_cumulativo))) - 1]
eixos[1, 1].annotate(
    f"Top 20% captura {captura_20:.0%}",
    xy=(0.20, captura_20),
    xytext=(0.35, 0.6),
    arrowprops=dict(arrowstyle="->", color="black")
)

# 6. Distribuição das probabilidades por classe real
eixos[1, 2].hist(y_proba_melhor[y_teste.to_numpy() == 0], bins=30, alpha=0.6, color="steelblue", label="Não Churn (real)")
eixos[1, 2].hist(y_proba_melhor[y_teste.to_numpy() == 1], bins=30, alpha=0.6, color="indianred", label="Churn (real)")
eixos[1, 2].axvline(limiar_melhor, color="black", lw=1, linestyle="--", label=f"Limiar = {limiar_melhor:.3f}")
eixos[1, 2].set_xlabel("Probabilidade prevista de churn")
eixos[1, 2].set_ylabel("Frequência")
eixos[1, 2].set_title("Distribuição das probabilidades")
eixos[1, 2].legend()
eixos[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nCaptura de churners no top 20% da base ordenada pelo score: {captura_20:.1%}")
print(f"Lift no top 20%: {captura_20 / 0.20:.2f}x")

> **Como ler a avaliação (M4)**
> - **Ordenação:** ROC-AUC de 0,830 e KS de 0,523 no teste, perto da validação cruzada (0,847): o modelo não decorou o
>   treino. O KS máximo cai no score de 0,48, perto do limiar.
> - **Sim/Não:** recall de 78% (pega 234 dos 301 que saíram) com precisão de 52%; F1 de 0,624, perto do treino (0,641).
> - **Calibração:** os pontos ficam abaixo da diagonal: o modelo prevê mais churn do que acontece (no grupo mais alto,
>   prevê cerca de 89% e saem cerca de 76%). É o efeito do balanceamento; não muda a ordem dos clientes, mas a
>   probabilidade deve ser lida como uma nota de risco, e não como a chance exata de sair (Brier de 0,172).
> - **Resposta:** sim; no teste, o modelo ordena e classifica quase como no treino.

In [ ]:
# 3.3 Consistência sem retreino: treino (fora da dobra), teste e base total
# ============================================================
# CONSISTÊNCIA DO MELHOR MODELO — SEM RETREINO
# ============================================================
# O modelo definido na base de treino é apenas aplicado às
# bases de teste e completa (treino + teste), para verificar
# a consistência dos seus resultados.
# Nenhum parâmetro é recalculado e nenhum retreino é feito.
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Modelo já treinado na base de treino — exatamente o mesmo
# escolhido na comparação (parâmetros congelados).
modelo_final = resultados[melhor_nome].best_estimator_

# Base completa (treino + teste juntos)
X_total = pd.concat([X_treino_limpo, X_teste_limpo], axis=0)
y_total = pd.concat([y_treino, y_teste], axis=0)


# ------------------------------------------------------------
# 1. APLICAR O MODELO ÀS BASES DE TESTE E COMPLETA
# ------------------------------------------------------------

# Base de teste (holdout — nunca vista no treinamento)
proba_teste_final = modelo_final.predict_proba(X_teste_limpo)[:, 1]
pred_teste_final = (proba_teste_final >= limiar_melhor).astype(int)

# Base completa (treino + teste juntos)
proba_total_final = modelo_final.predict_proba(X_total)[:, 1]
pred_total_final = (proba_total_final >= limiar_melhor).astype(int)


# ------------------------------------------------------------
# 2. MÉTRICAS DE CONSISTÊNCIA NAS TRÊS BASES
# ------------------------------------------------------------

# Treino: valores out-of-fold da comparação de modelos
# (estimativa honesta, sem o vício do próprio treino).
# Teste e total: calculados acima, apenas aplicando o modelo.

df_consistencia_final = pd.DataFrame([
    {
        "Base": "Treino (OOF)",
        "Precision": linha_melhor["Precision_Treino_OOF"],
        "Recall": linha_melhor["Recall_Treino_OOF"],
        "F1": linha_melhor["F1_Treino_OOF"]
    },
    {
        "Base": "Teste",
        "Precision": precision_score(y_teste, pred_teste_final, zero_division=0),
        "Recall": recall_score(y_teste, pred_teste_final, zero_division=0),
        "F1": f1_score(y_teste, pred_teste_final, zero_division=0)
    },
    {
        "Base": "Total (treino + teste)",
        "Precision": precision_score(y_total, pred_total_final, zero_division=0),
        "Recall": recall_score(y_total, pred_total_final, zero_division=0),
        "F1": f1_score(y_total, pred_total_final, zero_division=0)
    }
])

print("=" * 65)
print(f"CONSISTÊNCIA DO MELHOR MODELO — {melhor_nome} (sem retreino)")
print("=" * 65)

display(df_consistencia_final.round(4))


# ------------------------------------------------------------
# 3. MATRIZES DE CONFUSÃO (TESTE E TOTAL)
# ------------------------------------------------------------

print("\nMatriz de confusão — base de teste:")
print(confusion_matrix(y_teste, pred_teste_final))

print("\nMatriz de confusão — base total (treino + teste):")
print(confusion_matrix(y_total, pred_total_final))


# ------------------------------------------------------------
# 4. VEREDITO DA CONSISTÊNCIA
# ------------------------------------------------------------

# Amplitude (máximo − mínimo) de cada métrica entre as bases:
# quanto menor, mais consistente o modelo é entre treino,
# teste e total.

amplitudes = df_consistencia_final.drop(columns="Base").apply(
    lambda col: col.max() - col.min()
)

print("\nAmplitude (máximo − mínimo) de cada métrica entre as bases:")
for metrica, valor in amplitudes.items():
    print(f"  {metrica}: {valor:.4f}")

print("\nVeredito da consistência (limiar de referência: 0.10):")
for metrica, valor in amplitudes.items():
    situacao = "consistente" if valor < 0.10 else "instável"
    print(f"  {metrica}: amplitude {valor:.4f} — {situacao}")

> **Como ler a consistência:** precisão, recall e F1 variam no máximo 0,025 entre as três bases (o limite de referência
> é 0,10): o modelo é estável.

## 4. KS e lift: quem ligar primeiro

> **M5 — Se só der para ligar para parte da carteira, quanto ganhamos?**

### Teoria

- **Lift por decil:** ordeno os clientes do maior para o menor risco e divido em 10 grupos (D1 = 10% de maior risco).
  Lift = taxa de churn do decil ÷ taxa média. **Captura acumulada** = dos que saem, quantos estão até aquele decil.
- **KS:** a maior distância entre as curvas acumuladas da nota de quem sai e de quem fica; acima de 0,4 já é uma boa
  separação.

In [ ]:
# 4.1 Lift por decil do melhor modelo (base de teste)
# ------------------------------------------------------------
# LIFT POR DECIL DO MELHOR MODELO (base de teste)
# ------------------------------------------------------------

# Ordenar a base pelo score (maior risco primeiro), dividir
# em decis e comparar a taxa de churn de cada decil com a
# taxa média da base: o lift mostra quanto o modelo concentra
# os churners nos grupos mais arriscados — em um contexto real,
# é o ganho de priorizar a retenção pelos clientes mais arriscados.

df_lift = pd.DataFrame({
    "score": y_proba_melhor,
    "churn_real": y_teste.to_numpy()
}).sort_values("score", ascending=False).reset_index(drop=True)

# D1 = os 10% com maior score; D10 = os 10% com menor score
df_lift["decil"] = pd.qcut(
    df_lift.index + 1,
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
)

lift_decil = df_lift.groupby("decil", observed=True).agg(
    Clientes=("churn_real", "size"),
    Churners=("churn_real", "sum"),
    Taxa_Churn=("churn_real", "mean")
).reset_index()

taxa_base = y_teste.mean()

lift_decil["Lift"] = lift_decil["Taxa_Churn"] / taxa_base
lift_decil["Churners_Acum"] = lift_decil["Churners"].cumsum()
lift_decil["Captura_Acumulada"] = lift_decil["Churners_Acum"] / y_teste.sum()
lift_decil["Lift_Acumulado"] = (
    lift_decil["Captura_Acumulada"]
    / (lift_decil.index + 1)
    * 10
)

print("\n" + "=" * 65)
print(f"LIFT POR DECIL — {melhor_nome} (base de teste)")
print(f"Taxa média de churn da base: {taxa_base:.4f}")
print("=" * 65)

display(lift_decil.round(4))

print(f"\nLift no decil 1 (10% mais arriscados): {lift_decil.iloc[0]['Lift']:.2f}x")
print(f"Captura acumulada até o decil 2 (top 20%): {lift_decil.iloc[1]['Captura_Acumulada']:.1%}")
print(f"Lift acumulado no top 20%: {lift_decil.iloc[1]['Lift_Acumulado']:.2f}x")

In [ ]:
# 4.2 Gráficos de KS e lift acumulado
# ============================================================
# GRÁFICOS — ESTATÍSTICA KS E LIFT ACUMULADO
# (melhor modelo, base de teste)
# ============================================================

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. ESTATÍSTICA KS — distribuição do score por classe
# ------------------------------------------------------------
# As curvas mostram o % acumulado de clientes (eixo y) por
# faixa de score (eixo x) para churners e não churners.
# O KS é a maior distância vertical entre as duas curvas:
# quanto maior, melhor o modelo separa os dois grupos.

churn_real = y_teste.to_numpy()
proba_churn = np.sort(y_proba_melhor[churn_real == 1])
proba_nao_churn = np.sort(y_proba_melhor[churn_real == 0])

cdf_churn = np.arange(1, len(proba_churn) + 1) / len(proba_churn)
cdf_nao_churn = np.arange(1, len(proba_nao_churn) + 1) / len(proba_nao_churn)

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.5))
fig.suptitle(f"{melhor_nome} — base de teste", fontsize=14, y=1.02)

eixos[0].plot(
    proba_nao_churn, cdf_nao_churn,
    color="steelblue", lw=2.2, label="Não Churn"
)
eixos[0].plot(
    proba_churn, cdf_churn,
    color="indianred", lw=2.2, label="Churn"
)

# Ponto de máxima separação (KS) — exato via curva ROC:
# tpr − fpr num limiar t equivale à distância vertical entre
# as ECDFs em t (|F_churn − F_nao| = |tpr − fpr|).
from sklearn.metrics import roc_curve

fpr_ks, tpr_ks, thr_ks = roc_curve(y_teste, y_proba_melhor)
i_ks = int(np.argmax(tpr_ks - fpr_ks))
ks_grafico = float(tpr_ks[i_ks] - fpr_ks[i_ks])
score_ks = float(thr_ks[i_ks])
# Valores das ECDFs no ponto do KS (F = 1 − taxa acima do limiar)
y_ks_churn = float(1 - tpr_ks[i_ks])
y_ks_nao = float(1 - fpr_ks[i_ks])

eixos[0].plot(
    [score_ks] * 2, [y_ks_nao, y_ks_churn],
    color="black", lw=2.5, solid_capstyle="butt"
)
eixos[0].annotate(
    "KS = {:.3f}\n(score = {:.2f})".format(ks_grafico, score_ks),
    xy=(score_ks, (y_ks_nao + y_ks_churn) / 2),
    xytext=(0.08, 0.62), textcoords="axes fraction",
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=11, fontweight="bold"
)

eixos[0].set_title("Estatística KS — distribuição do score por classe")
eixos[0].set_xlabel("Probabilidade prevista de churn")
eixos[0].set_ylabel("% acumulado de clientes")
eixos[0].legend(loc="lower right")
eixos[0].grid(alpha=0.3)

# ------------------------------------------------------------
# 2. LIFT ACUMULADO — ordenando a base pelo score
# ------------------------------------------------------------
# Ordenamos a base do maior para o menor score e comparamos
# a captura de churners em cada ponto com a captura aleatória
# (lift = 1). É o ganho de priorizar a retenção pelos
# clientes mais arriscados.

ordem = np.argsort(-y_proba_melhor)
churn_ordenado = churn_real[ordem]

frac_base = np.arange(1, len(churn_ordenado) + 1) / len(churn_ordenado)
captura = np.cumsum(churn_ordenado) / churn_ordenado.sum()
lift_acum = captura / frac_base

# Índice do último cliente de cada decil (10%, 20%, ...)
idx_decis = (np.arange(1, 11) * len(churn_ordenado)) // 10 - 1

eixos[1].plot(
    frac_base * 100, lift_acum,
    color="darkorange", lw=2.2, label="Lift acumulado"
)
eixos[1].axhline(1, color="gray", lw=1.2, ls="--", label="Sem modelo (lift = 1)")

eixos[1].plot(
    frac_base[idx_decis] * 100, lift_acum[idx_decis],
    "o", color="black", ms=5
)
for i, idx in enumerate(idx_decis):
    eixos[1].annotate(
        f"D{i + 1}\n{lift_acum[idx]:.2f}x",
        (frac_base[idx] * 100, lift_acum[idx]),
        textcoords="offset points", xytext=(6, -2), fontsize=8
    )

# Destaque no top 20%
eixos[1].axvline(20, color="steelblue", ls=":", lw=1.5)
eixos[1].annotate(
    f"Top 20%: lift {lift_acum[idx_decis[1]]:.2f}x\n({captura[idx_decis[1]]:.0%} dos churners)",
    xy=(20, lift_acum[idx_decis[1]]),
    xytext=(35, lift_acum[idx_decis[0]]),
    arrowprops=dict(arrowstyle="->", color="steelblue"),
    fontsize=10, color="steelblue", fontweight="bold"
)

eixos[1].set_title("Lift acumulado — base ordenada pelo score")
eixos[1].set_xlabel("% da base priorizada (do mais arriscado para o menos)")
eixos[1].set_ylabel("Lift acumulado")
eixos[1].legend(loc="upper right")
eixos[1].grid(alpha=0.3)
eixos[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

# Conferência: o KS do gráfico deve bater com o da célula anterior
print(f"KS calculado no gráfico: {ks_grafico:.4f}")
print(f"KS da curva ROC (célula anterior): {ks:.4f}")

> **Como ler e resposta (M5)**
> - **D1:** 76% de churn, 2,85 vezes a média.
> - **Ganho:** ligando para os **20% de maior risco**, o time encontra **48% dos clientes que vão sair** (lift acumulado de
>   2,41); com 40%, encontra 78%. Nos 10% de menor risco, saem só 2%.
> - **Para o negócio:** se o time só consegue ligar para N clientes, liga para os N de maior nota.

## 5. Leitura de negócio

> **M6 — O que mais aumenta e diminui o risco de sair?**

Os coeficientes da logística (com as variáveis padronizadas) dizem a direção e o peso de cada variável; agrupados em
temas, mostram o que o negócio pode atacar.

In [ ]:
# 5.1 Tabelas de previsões: base de teste e base total
# ============================================================
# TABELAS DE PREVISÕES
# 1) Base de teste (holdout)
# 2) Base TOTAL (treino + teste juntos) com o churn previsto
#    e a sua probabilidade
# ============================================================

# O modelo permanece treinado apenas na base de treino:
# as previsões abaixo são apenas a aplicação dele (sem retreino).

# ------------------------------------------------------------
# 1. Tabela da base de teste
# ------------------------------------------------------------

tabela_final = pd.DataFrame({
    "customerID": X_teste_original["customerID"].to_numpy(),
    "Churn_Real": y_teste.to_numpy(),
    "Probabilidade_Churn": y_proba_melhor,
    "Churn_Previsto": y_pred_melhor
}).sort_values("Probabilidade_Churn", ascending=False).reset_index(drop=True)

display(tabela_final)

print("\nResumo (base de teste):")
print("Registros:", len(tabela_final))
print("Taxa real de churn na base de teste:", round(tabela_final["Churn_Real"].mean(), 4))
print("Taxa prevista de churn (limiar otimizado):", round(tabela_final["Churn_Previsto"].mean(), 4))
print("Acordo entre previsão e real:", round((tabela_final["Churn_Previsto"] == tabela_final["Churn_Real"]).mean(), 4))


# ------------------------------------------------------------
# 2. Tabela TOTAL (treino + teste juntos)
# ------------------------------------------------------------

# Probabilidades nas duas bases, geradas pelo mesmo modelo
# treinado na base de treino, no mesmo limiar otimizado.

proba_treino = melhor_modelo.predict_proba(X_treino_limpo)[:, 1]
proba_teste = y_proba_melhor

tabela_total = pd.DataFrame({
    "customerID": pd.concat(
        [X_treino_original["customerID"], X_teste_original["customerID"]],
        axis=0
    ).to_numpy(),
    "Base": (
        ["Treino"] * len(X_treino_limpo)
        + ["Teste"] * len(X_teste_limpo)
    ),
    "Churn_Real": pd.concat([y_treino, y_teste], axis=0).to_numpy(),
    "Probabilidade_Churn": np.concatenate([proba_treino, proba_teste])
})

# Churn previsto no limiar otimizado (definido na base de treino)
tabela_total["Churn_Previsto"] = (
    tabela_total["Probabilidade_Churn"] >= limiar_melhor
).astype(int)

# Ordenar pelo risco (maior probabilidade primeiro)
display(
    tabela_total
    .sort_values("Probabilidade_Churn", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 65)
print("RESUMO DA TABELA TOTAL (TREINO + TESTE JUNTOS)")
print("=" * 65)

print("Registros:", len(tabela_total))
print("Taxa real de churn:", round(tabela_total["Churn_Real"].mean(), 4))
print("Taxa prevista de churn (limiar otimizado):", round(tabela_total["Churn_Previsto"].mean(), 4))
print("Acordo entre previsão e real:", round((tabela_total["Churn_Previsto"] == tabela_total["Churn_Real"]).mean(), 4))

# Resumo por base (treino vs teste), para acompanhar a consistência
resumo_por_base = tabela_total.groupby("Base").agg(
    Registros=("Churn_Real", "size"),
    Taxa_Real=("Churn_Real", "mean"),
    Taxa_Prevista=("Churn_Previsto", "mean"),
    Probabilidade_Media=("Probabilidade_Churn", "mean")
).round(4)

display(resumo_por_base)

In [ ]:
# 5.2 Importância das variáveis e peso de cada tema de negócio
# ============================================================
# ANÁLISE DAS VARIÁVEIS — ALINHAMENTO COM O NEGÓCIO
# ============================================================
# Contexto: prever o churn de clientes de uma empresa de
# telemarketing/telecom. Antes de seguir para a etapa final,
# revisamos o percurso das variáveis (da base original até a
# ABT final) e verificamos se as 41 variáveis levadas aos
# modelos contam uma história consistente com esse objetivo.
# ============================================================

# ------------------------------------------------------------
# 1. O PERCURSO DAS VARIÁVEIS
# ------------------------------------------------------------

print("=" * 65)
print("PERCURSO DAS VARIÁVEIS")
print("=" * 65)

print("Colunas na base original (treino):", treino.shape[1])
print(
    "Variáveis categóricas redundantes removidas na limpeza:",
    len(variaveis_redundantes)
)
print(
    "Dummies criadas na preparação; variáveis preditoras analisadas:",
    df_modelagem.shape[1] - 2  # menos customerID e Churn
)
print(
    "Variáveis retiradas na análise (redundância/colinearidade/não significância):",
    len(variaveis_retirar)
)
print("Variáveis finais levadas aos modelos:", X_treino_limpo.shape[1])


# ------------------------------------------------------------
# 2. IMPORTÂNCIA E DIREÇÃO PELO MELHOR MODELO
# ------------------------------------------------------------

# Para modelos lineares usamos o coeficiente (direção do efeito
# sobre o churn); para modelos de árvore usamos a importância.

modelo_melhor_step = resultados[melhor_nome].best_estimator_.named_steps["modelo"]

if hasattr(modelo_melhor_step, "coef_"):
    coeficientes = modelo_melhor_step.coef_[0]
    df_importancia = pd.DataFrame({
        "Variavel": X_treino_limpo.columns,
        "Importancia": np.abs(coeficientes),
        "Efeito": np.where(
            coeficientes > 0,
            "Aumenta o risco de churn",
            "Reduz o risco de churn"
        )
    })
    tipo_importancia = "coeficiente (modelo linear)"
elif hasattr(modelo_melhor_step, "feature_importances_"):
    importancias = modelo_melhor_step.feature_importances_
    df_importancia = pd.DataFrame({
        "Variavel": X_treino_limpo.columns,
        "Importancia": importancias,
        "Efeito": "—"  # árvores não dão direção direta
    })
    tipo_importancia = "feature importance (modelo de árvore)"
else:
    raise ValueError("O melhor modelo não expõe coef_ nem feature_importances_.")

print("\n" + "=" * 65)
print(f"IMPORTÂNCIA DAS VARIÁVEIS — {melhor_nome} ({tipo_importancia})")
print("=" * 65)

df_importancia = df_importancia.sort_values(
    "Importancia",
    ascending=False
).reset_index(drop=True)

display(df_importancia.head(15).round(4))


# ------------------------------------------------------------
# 3. VISÃO POR TEMA DE NEGÓCIO
# ------------------------------------------------------------

# As 41 variáveis finais agrupadas nos grandes temas que
# explicam o churn de uma operação de telemarketing/telecom:

TEMAS_VARIAVEIS = {
    "Fidelidade / tempo de casa": [
        "tenure", "TenureCohort_6-12", "TenureCohort_13-24",
        "TenureCohort_25+", "is_new_customer", "new_monthly_customer"
    ],
    "Contrato": [
        "Contract_One year", "Contract_Two year", "IsHighRiskContract"
    ],
    "Serviços e pacotes": [
        "InternetService_Fiber optic", "InternetService_No",
        "MultipleLines_Yes", "MultipleLines_No phone service",
        "n_streaming", "n_protecao", "OnlineSecurity_bin",
        "OnlineBackup_bin", "DeviceProtection_bin", "TechSupport_bin",
        "StreamingTV_bin", "StreamingMovies_bin",
        "TotalServicesSubscribed", "no_support_with_internet",
        "HasFiberOpticWithoutTechSupport", "mensalidade_relativa_pacote"
    ],
    "Pagamento": [
        "PaymentMethod_Electronic check", "PaymentMethod_Mailed check",
        "PaymentMethod_Credit card (automatic)", "PaperlessBilling_Yes",
        "pagamento_automatico"
    ],
    "Financeiro": [
        "MonthlyCharges", "TotalCharges", "ChargesRatio",
        "charge_difference", "fl_total_charges_ausente"
    ],
    "Perfil do cliente": [
        "SeniorCitizen", "Partner_Yes", "Dependents_Yes",
        "has_family", "MonthlyIncome", "IncomeCommitment"
    ]
}

# Conferência: toda variável final pertence a exatamente um tema
variavel_tema = {
    var: tema
    for tema, vars_tema in TEMAS_VARIAVEIS.items()
    for var in vars_tema
}

sem_tema = set(X_treino_limpo.columns) - set(variavel_tema)
assert len(sem_tema) == 0, f"Variáveis sem tema definido: {sem_tema}"
assert len(variavel_tema) == X_treino_limpo.shape[1]

df_importancia["Tema"] = df_importancia["Variavel"].map(variavel_tema)

resumo_temas = df_importancia.groupby("Tema").agg(
    Qtd_Variaveis=("Variavel", "size"),
    Importancia_Total=("Importancia", "sum"),
    Importancia_Media=("Importancia", "mean"),
    Mais_Relevante=("Importancia", "max")
).sort_values("Importancia_Total", ascending=False)

# % da importância do modelo concentrada em cada tema
resumo_temas["Peso_do_Tema"] = (
    resumo_temas["Importancia_Total"] / resumo_temas["Importancia_Total"].sum()
)

print("\n" + "=" * 65)
print("IMPORTÂNCIA POR TEMA DE NEGÓCIO")
print("=" * 65)

display(resumo_temas.round(4))

# Top variável de cada tema, com o efeito (quando disponível)
top_por_tema = (
    df_importancia
    .sort_values("Importancia", ascending=False)
    .groupby("Tema")
    .head(3)
    .sort_values(["Tema", "Importancia"], ascending=[True, False])
    [["Tema", "Variavel", "Importancia", "Efeito"]]
)

print("\nPrincipais variáveis de cada tema:")
display(top_por_tema.round(4))

print(
    "\nConclusão: as", X_treino_limpo.shape[1],
    "variáveis finais cobrem os seis temas centrais do negócio"
    " (fidelidade, contrato, serviços, pagamento, financeiro e perfil),"
    " sem variáveis redundantes — a mesma história que o modelo usa"
    " para ranquear o risco de churn."
)

> **Como ler e resposta (M6)**
> - **Peso dos temas:** fidelidade e tempo de casa (32%), serviços e pacotes (26%), contrato (20%), financeiro (10%),
>   pagamento (7%) e perfil (4%). Tempo de casa e contrato somam mais da metade, com 9 das 41 variáveis.
> - **Aumentam o risco:** cheque eletrônico, internet sem suporte técnico, fibra e cliente idoso.
> - **Reduzem:** contrato de dois anos e de um ano, tempo de casa e segurança online.
> - **Preço e serviços se leem juntos:** `MonthlyCharges` (+) e `mensalidade_relativa_pacote` (−), `n_streaming` e
>   `MultipleLines_Yes` andam com a fibra; um por um, os sinais não têm leitura de negócio. Coeficientes pequenos (menos
>   de 0,1), como `IsHighRiskContract`, só ajustam o modelo depois de descontadas as outras variáveis.
> - **A história:** quem fica tem casa longa e contrato anual ou de dois anos; quem sai é novo, mensal, paga por cheque
>   eletrônico e tem fibra sem suporte. O risco vem mais do que o cliente contrata e de como paga do que de quem ele é.

## 6. Produção: pipeline, previsões e MLflow

> **M7 — Quantos clientes da carteira estão em risco, e onde está o resultado?**

### Teoria: o pipeline de produção

Todas as limpezas feitas depois da importação (retirar as redundantes, criar as dummies, retirar as 12 variáveis,
alinhar as 41 colunas) vão para **um único objeto**, junto com o modelo já treinado. Ele recebe a base no formato da
ABT e devolve a probabilidade. A célula 6.1 confere que o pipeline reproduz **exatamente** as bases e as probabilidades
da modelagem. O modelo **não é retreinado**: é o mesmo definido no treino.

In [ ]:
# 6.1 Pipeline de produção (limpeza + modelo) e validação
# ============================================================
# PIPELINE DE PRODUÇÃO — LIMPEZA + MODELO EM UM ÚNICO OBJETO
# ============================================================
# Todas as limpezas feitas após a importação das bases foram
# empacotadas num único Pipeline sklearn, que recebe a base
# crua no formato da ABT e devolve a probabilidade de churn:
#
#   1. Remoção das variáveis categóricas redundantes
#   2. Dummies das variáveis categóricas (drop_first=True)
#   3. Seleção das colunas numéricas (sem customerID/Churn)
#   4. Remoção das variáveis retiradas na análise
#   5. Coerção numérica + infinitos → NaN
#   6. Alinhamento exato nas 41 colunas do modelo
#   7. Modelo (imputer + scaler + regressão logística)
#
# O modelo continua treinado apenas na base de treino
# (parâmetros congelados, sem retreino). Este é o objeto de
# produção: basta chamar modelo_producao.predict_proba(base)
# com a base crua no formato da ABT.
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

colunas_modelo = X_treino_limpo.columns.tolist()


def preparar_base(df):
    """Reproduz exatamente as limpezas feitas após a importação.

    Recebe a base crua no formato da ABT (com ou sem Churn) e
    devolve exatamente as 41 colunas que o modelo espera.
    """
    df = df.copy()

    # 1. Remover as variáveis categóricas redundantes
    #    (mesma lista da etapa de limpeza)
    df = df.drop(columns=variaveis_redundantes, errors="ignore")

    # 2. Dummies com as MESMAS variáveis categóricas da
    #    preparação (drop_first=True)
    df = pd.get_dummies(
        df,
        columns=variaveis_categoricas,
        drop_first=True,
        dtype=float
    )

    # 3. Selecionar numéricas e dummies, sem ID e sem alvo
    X = df.select_dtypes(include=["number", "bool"])
    X = X.drop(columns=["customerID", "Churn"], errors="ignore")

    # 4. Remover as variáveis retiradas na análise
    #    (redundância/colinearidade/não significância)
    X = X.drop(columns=variaveis_retirar, errors="ignore")

    # 5. Coerção numérica e infinitos → NaN
    #    (mesma etapa aplicada na modelagem; nulos são
    #    imputados pelo SimpleImputer dentro do modelo)
    X = X.apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )

    # 6. Exatamente as colunas do modelo, na mesma ordem
    #    (dummies ausentes na base nova entram como 0)
    X = X.reindex(columns=colunas_modelo, fill_value=0.0)

    return X.astype(float)


# Pipeline único: limpeza + modelo (treinado na base de treino)
modelo_producao = Pipeline([
    ("limpeza", FunctionTransformer(preparar_base)),
    ("modelo", modelo_final)
])

print(modelo_producao)


# ------------------------------------------------------------
# 1. VALIDAÇÃO — o pipeline reproduz a modelagem?
# ------------------------------------------------------------

# A limpeza do pipeline aplicada à base original tem que
# devolver EXATAMENTE as bases usadas no treino e no teste.
X_validacao = modelo_producao.named_steps["limpeza"].transform(treino)

check_treino = np.allclose(
    X_validacao.loc[X_treino_limpo.index].to_numpy(),
    X_treino_limpo.to_numpy()
)
check_teste = np.allclose(
    X_validacao.loc[X_teste_limpo.index].to_numpy(),
    X_teste_limpo.to_numpy()
)

print("\nReproduz exatamente a base de treino:", check_treino)
print("Reproduz exatamente a base de teste:", check_teste)

assert check_treino, "A limpeza do pipeline difere da base de treino."
assert check_teste, "A limpeza do pipeline difere da base de teste."

# As probabilidades do pipeline batem com as da avaliação?
# O pipeline espera a base CRUA no formato da ABT — X_teste_original
# já tem dummies, então usamos as linhas cruas correspondentes
# ao conjunto de teste.
proba_check = modelo_producao.predict_proba(
    treino.loc[X_teste_original.index]
)[:, 1]
check_proba = np.allclose(proba_check, y_proba_melhor)

print("Probabilidades idênticas às da avaliação:", check_proba)

assert check_proba, "As probabilidades do pipeline diferem da avaliação."

print(
    "\nPipeline validado: recebendo a base crua, ele reproduz"
    " a limpeza e as previsões da etapa de modelagem."
)

In [ ]:
# 6.2 Base de inferência pontuada pelo pipeline e distribuição por decil
# ------------------------------------------------------------
# 2. CONTEXTO REAL DE PRODUÇÃO — CLIENTES NOVOS
# ------------------------------------------------------------
# A base de inferência representa clientes novos, no formato
# da ABT. Não há separação treino/teste aqui: é um cenário
# real, no qual apenas passamos a base crua pelo pipeline
# (limpeza + previsão) — sem retreino.

base_inferencia = inferencia.copy()

probas_inferencia = modelo_producao.predict_proba(base_inferencia)[:, 1]
pred_inferencia = (probas_inferencia >= limiar_melhor).astype(int)

tabela_inferencia = pd.DataFrame({
    "customerID": base_inferencia["customerID"].to_numpy(),
    "Probabilidade_Churn": probas_inferencia,
    "Churn_Previsto": pred_inferencia
}).sort_values("Probabilidade_Churn", ascending=False).reset_index(drop=True)

print("\n" + "=" * 65)
print(f"BASE DE INFERÊNCIA PONTUADA PELO PIPELINE — {melhor_nome}")
print("=" * 65)

display(tabela_inferencia)

print("\nResumo da inferência:")
print("Registros:", len(tabela_inferencia))
print("Taxa prevista de churn (limiar otimizado):", round(pred_inferencia.mean(), 4))

n_top = int(len(tabela_inferencia) * 0.20)
print(f"Top 20% mais arriscados: {n_top} clientes")


# ------------------------------------------------------------
# 3. DISTRIBUIÇÃO POR DECIL DE RISCO
# ------------------------------------------------------------

# Sem rótulo real, mostramos apenas a concentração do risco:
# quantos clientes e quantos churners previstos há em cada
# decil de probabilidade — a fila de priorização da retenção.

tabela_inferencia["decil"] = pd.qcut(
    tabela_inferencia["Probabilidade_Churn"].rank(method="first", ascending=False),
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
)

distribuicao_decil = tabela_inferencia.groupby("decil", observed=True).agg(
    Clientes=("Probabilidade_Churn", "size"),
    Churn_Previsto=("Churn_Previsto", "sum"),
    Probabilidade_Media=("Probabilidade_Churn", "mean")
).reset_index()

print("\n" + "=" * 65)
print("DISTRIBUIÇÃO POR DECIL DE RISCO — BASE DE INFERÊNCIA")
print("=" * 65)

display(distribuicao_decil.round(4))

churn_decil_1 = distribuicao_decil.loc[0, "Churn_Previsto"]
print(
    f"\nDecil 1 (10% mais arriscados): {churn_decil_1} clientes previstos como churn,"
    f" {churn_decil_1 / max(pred_inferencia.sum(), 1):.1%} de todos os churners previstos."
)

In [ ]:
# 6.3 Publica as tabelas no Catálogo
# ============================================================
# PUBLICAÇÃO DAS TABELAS NO UNITY CATALOG
# ============================================================
# Publicamos duas tabelas no mesmo esquema das bases de
# origem (workspace.terra_signal), para o grupo inteiro
# conseguir consultar no Catalog:
#
#   1) gold_churn_abt_scored — a ABT completa (todas as
#      colunas originais) com a nossa estimativa
#      (Churn_Previsto), a probabilidade de churn e o
#      decil de risco — além da marcação Treino/Teste.
#
#   2) gold_churn_inferencia_scored — a base de inferência
#      (clientes novos) pontuada pelo pipeline de produção,
#      com probabilidade, previsão e decil de risco.
# ============================================================

# ------------------------------------------------------------
# 1. ABT completa + estimativa e probabilidade
# ------------------------------------------------------------

df_abt_scored = treino.copy()

# O pipeline recebe a base crua e faz a limpeza internamente
proba_abt = modelo_producao.predict_proba(df_abt_scored)[:, 1]

df_abt_scored["Probabilidade_Churn"] = proba_abt
df_abt_scored["Churn_Previsto"] = (proba_abt >= limiar_melhor).astype(int)

# Marcar de qual parte do holdout cada linha veio (as linhas
# de treino têm previsão in-sample; as de teste, out-of-sample)
df_abt_scored["Base_Modelagem"] = np.where(
    df_abt_scored.index.isin(X_treino_original.index),
    "Treino",
    "Teste"
)

# Decil de risco (D1 = 10% mais arriscados)
df_abt_scored["Decil_Risco"] = pd.qcut(
    pd.Series(proba_abt).rank(method="first", ascending=False),
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
).astype(str)

print("Tabela 1 — ABT + estimativa e probabilidade:", df_abt_scored.shape)

display(df_abt_scored.head())

spark.createDataFrame(df_abt_scored).write.mode("overwrite").saveAsTable(
    "workspace.terra_signal.gold_churn_abt_scored"
)

print("\nSalva: workspace.terra_signal.gold_churn_abt_scored")
print("Registros no catálogo:", spark.table(
    "workspace.terra_signal.gold_churn_abt_scored"
).count())


# ------------------------------------------------------------
# 2. Base de inferência pontuada (clientes novos)
# ------------------------------------------------------------

df_inferencia_scored = inferencia.copy()

proba_inf = modelo_producao.predict_proba(df_inferencia_scored)[:, 1]

df_inferencia_scored["Probabilidade_Churn"] = proba_inf
df_inferencia_scored["Churn_Previsto"] = (proba_inf >= limiar_melhor).astype(int)
df_inferencia_scored["Decil_Risco"] = pd.qcut(
    pd.Series(proba_inf).rank(method="first", ascending=False),
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
).astype(str)

print("Tabela 2 — base de inferência pontuada:", df_inferencia_scored.shape)

display(df_inferencia_scored.head())

spark.createDataFrame(df_inferencia_scored).write.mode("overwrite").saveAsTable(
    "workspace.terra_signal.gold_churn_inferencia_scored"
)

print("\nSalva: workspace.terra_signal.gold_churn_inferencia_scored")
print("Registros no catálogo:", spark.table(
    "workspace.terra_signal.gold_churn_inferencia_scored"
).count())

In [ ]:
# 6.4 Registra o pipeline no MLflow
# ============================================================
# MODELO PRONTO — REGISTRO NO MLFLOW
# ============================================================
# O pipeline de produção completo (limpeza + modelo) é
# registrado no MLflow com assinatura e exemplo de entrada,
# pronto para ser carregado em qualquer ambiente com:
#
#   mlflow.pyfunc.load_model(model_uri)
#
# Nenhum retreino é feito: o modelo é o mesmo definido na
# base de treino, com parâmetros congelados.
# ============================================================

import mlflow
from mlflow.models import infer_signature

# ajuste do grupo: experimento com nome fixo. O experimento padrão é o do notebook e vai para a lixeira
# junto com ele (ao apagar ou reimportar o 06); com nome fixo, o registro não depende do arquivo do notebook.
EXPERIMENTO = "/Shared/terra_signal_churn"
mlflow.set_experiment(EXPERIMENTO)

# Exemplo de entrada: a base CRUA no formato da ABT
X_exemplo = treino.drop(columns=["Churn"]).head(10)
pred_exemplo = modelo_producao.predict_proba(X_exemplo)[:, 1]

# A assinatura usa as previsões do próprio modelo como saída
signature = infer_signature(
    model_input=X_exemplo,
    model_output=pd.DataFrame({"Probabilidade_Churn": pred_exemplo})
)

with mlflow.start_run(run_name="modelo_producao_churn") as run:

    # Parâmetros congelados do melhor modelo
    # ajuste do grupo: lidos do modelo escolhido na 2.9, e não escritos à mão
    p_modelo = modelo_final.named_steps["modelo"].get_params()
    mlflow.log_params({
        "modelo": melhor_nome,
        **{k: p_modelo[k] for k in ["C", "penalty", "solver", "class_weight"] if k in p_modelo},
        **{k.replace("modelo__", ""): v for k, v in resultados[melhor_nome].best_params_.items()},
        "limiar_decisao": round(limiar_melhor, 4),
        "n_variaveis": int(X_treino_limpo.shape[1]),
        "cv_folds": 5
    })

    # Métricas na base de teste (holdout)
    mlflow.log_metrics({
        "precision_teste": precision_score(y_teste, y_pred_melhor),
        "recall_teste": recall_score(y_teste, y_pred_melhor),
        "f1_teste": f1_score(y_teste, y_pred_melhor),
        "roc_auc_teste": roc_auc_score(y_teste, y_proba_melhor),
        "pr_auc_teste": average_precision_score(y_teste, y_proba_melhor)
    })

    # Log do pipeline completo (limpeza + modelo)
    try:
        model_info = mlflow.sklearn.log_model(
            modelo_producao,
            name="pipeline_churn",
            signature=signature,
            input_example=X_exemplo
        )
    except TypeError:
        # Compatibilidade com MLflow 2.x
        model_info = mlflow.sklearn.log_model(
            modelo_producao,
            artifact_path="pipeline_churn",
            signature=signature,
            input_example=X_exemplo
        )

print("=" * 65)
print("MODELO REGISTRADO NO MLFLOW")
print("=" * 65)
print("Run ID:", run.info.run_id)
print("Experimento:", EXPERIMENTO, f"(id {run.info.experiment_id})")
print("Model URI:", model_info.model_uri)
print("\nO pipeline completo (limpeza + modelo) pode ser carregado com:")
print(f"  mlflow.pyfunc.load_model('{model_info.model_uri}')")

> **Como ler e resposta (M7)**
> - **Carteira:** 571 dos 1.409 clientes (40,5%) ficam acima do limiar; o decil 1 tem 141 clientes, com probabilidade
>   média de 0,89, e os decis 1 a 4 concentram todos os "Yes" menos 7.
> - **Onde está:** `workspace.terra_signal.gold_churn_abt_scored` (5.634 clientes do histórico) e
>   `workspace.terra_signal.gold_churn_inferencia_scored` (1.409 da carteira), com `Probabilidade_Churn`,
>   `Churn_Previsto` e `Decil_Risco`; o pipeline está no MLflow (registro `dabb7c50…`, Lasso com C = 0,3) e carrega com
>   `mlflow.pyfunc.load_model`.
> - **Quem usa:** o 07 (segmentação) e o 08 (arquivo da entrega) leem a `gold_churn_inferencia_scored`.

In [ ]:
# Respostas às perguntas de negócio, a partir dos resultados deste notebook
lm = linha_melhor
# M6: as variáveis de preço e de quantidade de serviços carregam o tipo de internet (multicolinearidade, 5.5c do 05);
# os sinais delas, um por um, não têm leitura de negócio, então ficam fora da lista e entram como nota
LER_EM_CONJUNTO = ["MonthlyCharges", "mensalidade_relativa_pacote", "n_streaming", "MultipleLines_Yes", "IsHighRiskContract"]
imp_leitura = df_importancia[~df_importancia["Variavel"].isin(LER_EM_CONJUNTO)]
RESPOSTAS = {
    "M1": f"{X_treino_limpo.shape[1]} variáveis: saem as 6 categóricas redundantes e {len(variaveis_retirar)} variáveis por duplicata exata, "
          "versão repetida do tempo de casa ou falta de associação (gender, após a correção FDR)",
    "M2": f"{melhor_nome}, escolhido pela consistência entre treino, teste e base total (ROC-AUC de {virg(lm['ROC_AUC_Teste'], 3)} no teste)",
    "M3": f"Limiar de {virg(limiar_melhor, 4)}, o que maximiza o F1 nas probabilidades fora da dobra do treino",
    "M4": f"Sim: no teste, F1 de {virg(lm['F1_Teste'], 3)}, recall de {virg(100 * lm['Recall_Teste'], 0)}% e precisão de {virg(100 * lm['Precision_Teste'], 0)}%, "
          f"no mesmo nível do treino (F1 {virg(lm['F1_Treino_OOF'], 3)})",
    "M5": f"Os 20% de maior risco concentram {virg(100 * lift_decil.iloc[1]['Captura_Acumulada'], 0)}% dos churners (lift acumulado de {virg(lift_decil.iloc[1]['Lift_Acumulado'], 2)})",
    "M6": "Aumentam: " + ", ".join(imp_leitura.loc[imp_leitura["Efeito"].str.startswith("Aumenta"), "Variavel"].head(4)) +
          ". Reduzem: " + ", ".join(imp_leitura.loc[imp_leitura["Efeito"].str.startswith("Reduz"), "Variavel"].head(4)) +
          ". Preço e número de serviços andam junto com a fibra e se leem em conjunto, e não um por um (seção 5)",
    "M7": f"{int(pred_inferencia.sum())} de {virg(len(pred_inferencia), 0)} clientes ({virg(100 * pred_inferencia.mean(), 1)}%) acima do limiar; "
          "tabelas gold_churn_abt_scored e gold_churn_inferencia_scored, e o pipeline no MLflow",
}
PERGUNTAS = {"M1": "Que variáveis entram no modelo, e por quê?", "M2": "Qual algoritmo prevê melhor o churn?",
             "M3": "A partir de que probabilidade dizemos que o cliente vai sair?", "M4": "O modelo funciona em clientes que ele não viu?",
             "M5": "Se só der para ligar para parte da carteira, quanto ganhamos?", "M6": "O que mais aumenta e diminui o risco de sair?",
             "M7": "Quantos clientes da carteira estão em risco, e onde está o resultado?"}
display(pd.DataFrame({"pergunta": PERGUNTAS, "resposta": RESPOSTAS}).rename_axis("#").reset_index())

In [ ]:
# Grava o resumo do modelo escolhido, para o 07 e o 08 lerem (em vez de copiar os números à mão)
lm = linha_melhor
info = pd.DataFrame([{
    "modelo": melhor_nome, "limiar": float(limiar_melhor), "n_variaveis": int(X_treino_limpo.shape[1]),
    "roc_auc_teste": float(lm["ROC_AUC_Teste"]), "f1_teste": float(lm["F1_Teste"]),
    "recall_teste": float(lm["Recall_Teste"]), "precision_teste": float(lm["Precision_Teste"]),
    "parametros": str(modelo_final.get_params().get("modelo", modelo_final)),
    # endereço do registro no MLflow (6.4): o 09 e o 10 leem daqui, em vez de copiar o ID à mão
    "model_uri": model_info.model_uri if "model_info" in globals() else None,
    "run_id": run.info.run_id if "run" in globals() else None,
}])
spark.createDataFrame(info).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_churn_modelo_info")
display(info)

## Checklist final do 06

| # | Decisão | Status | Onde |
|---|---|---|---|
| D21 | Modelo final | ☑ Logística Lasso, 41 colunas, C = 0,3, classes balanceadas; AUC 0,830 no teste (11/11) | Notebook 06 |
| D22 | Limiar do Sim/Não: 0,5046, maior F1 nas probabilidades fora da dobra do treino | ☑ Decidido (11/11) | Seção 2.7 |
| D19 | Seleção de variáveis | ☑ As 41 colunas do 05, reconstruídas e conferidas | Seção 1 |
| D23 | Arquivo da entrega | ☑ Gerado no 08 a partir de `gold_churn_inferencia_scored`: 571 de 1.409 com Yes (11/11) | Notebook 08 |

**Registro no MLflow:** Run ID `dabb7c50091f4264b45767f994fb6f29`; modelo `models:/m-2dd8f256aa384a98bf46d77f84ff80eb`.

**Limites a ter em mente:**
- **A probabilidade é uma nota de risco:** com o balanceamento, a média prevista fica perto de 41% para 27% de churn
  real. Para ordenar a fila e para o Sim/Não, isso não muda nada; para números de negócio que multiplicam a
  probabilidade (perda esperada), vale recalibrar antes.
- **Ordem das linhas:** resolvida. O 05 ordena o treino por `customerID` e grava a separação; este notebook
  a reconstrói, e os números se repetem em qualquer execução.

**Saída do 06:** `gold_churn_abt_scored`, `gold_churn_inferencia_scored`, `gold_churn_modelo_info` e o pipeline no MLflow.
**Próximos passos:** 07 (segmentação) e 08 (entrega).